In [ ]:
"""
============================================================
  SavvyMart — Roman Urdu Semantic Search Fine-Tuning
  Dataset : roman_urdu_dataset_v3.xlsx  (3,706 rows)
  Model   : paraphrase-multilingual-MiniLM-L12-v2
  Categories: Electronics, Bags, Clothes, Shoes, Hair Oil
============================================================

STEP 1 — Install dependencies (run once):
    pip install "accelerate>=0.26.0" --upgrade
    pip install "transformers[torch]" --upgrade
    pip install sentence-transformers datasets pandas openpyxl scikit-learn torch

STEP 2 — Place these files in the same folder:
    roman_urdu_dataset_v3.xlsx
    finetune_roman_urdu.py

STEP 3 — Run:
    python finetune_roman_urdu.py
============================================================
"""

import os, math, json, random, warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
    util,
)
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
#  CONFIG  — tweak as needed
# ─────────────────────────────────────────────────────────────
CONFIG = {
    "base_model"   : "paraphrase-multilingual-MiniLM-L12-v2",

    # ← updated to v3 dataset
    "dataset_path" : "roman_urdu_dataset_v3.xlsx",
    "output_dir"   : "savvymart_model",
    "log_path"     : "training_log.json",

    # With 3706 rows we don't need as many epochs — 5 is enough
    "epochs"       : 5,
    # Larger batch = more negatives per step = better MNRL training
    "batch_size"   : 32,
    "warmup_ratio" : 0.1,
    "learning_rate": 2e-5,
    "eval_split"   : 0.15,
    "seed"         : 42,

    # Dataset is already large — only light augmentation needed
    "augment_data" : True,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])


# ─────────────────────────────────────────────────────────────
#  1. LOAD DATASET
# ─────────────────────────────────────────────────────────────
def load_dataset_from_excel(path: str) -> pd.DataFrame:
    print("\n  Loading dataset...")
    if not Path(path).exists():
        raise FileNotFoundError(
            f"\n  ERROR: '{path}' not found!\n"
            f"  Make sure roman_urdu_dataset_v3.xlsx is in the same folder."
        )
    df = pd.read_excel(path, sheet_name="Full Dataset")
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    df = df.dropna(subset=["roman_urdu_query", "english_translation"])

    print(f"  Loaded  : {len(df)} rows")
    for cat, n in df["category"].value_counts().items():
        print(f"           {cat:<15} {n} rows")
    return df


# ─────────────────────────────────────────────────────────────
#  2. LIGHT AUGMENTATION
#     Dataset is already 3706 rows so we keep this minimal.
#     We only add English→English self-pairs so the model
#     also learns that identical English phrases are similar.
# ─────────────────────────────────────────────────────────────
def augment_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    # Primary pairs: Roman Urdu query → English translation
    for _, r in df.iterrows():
        rows.append({
            "anchor"  : r["roman_urdu_query"],
            "positive": r["english_translation"],
        })

    if not CONFIG["augment_data"]:
        return pd.DataFrame(rows)

    # Self-pairs: English → English
    # Teaches the model that English product terms map to themselves
    for _, r in df.iterrows():
        rows.append({
            "anchor"  : r["english_translation"],
            "positive": r["english_translation"],
        })

    # Spelling variants common in Roman Urdu ecommerce searches
    VARIANTS = {
        "kala"   : ["kali", "kaala"],
        "safaid" : ["safed", "sufaid"],
        "sasta"  : ["sasti", "saста"],
        "acha"   : ["achi", "accha"],
        "naya"   : ["nayi", "nai"],
        "bara"   : ["bada", "bari"],
        "chota"  : ["choti", "chhota"],
        "nila"   : ["nili", "neela"],
        "hara"   : ["hari", "haraa"],
        "lal"    : ["laal"],
    }
    for _, r in df.iterrows():
        q  = r["roman_urdu_query"]
        en = r["english_translation"]
        for word, alts in VARIANTS.items():
            if word in q:
                for alt in alts:
                    rows.append({
                        "anchor"  : q.replace(word, alt),
                        "positive": en,
                    })
                break  # one variant group per query

    aug_df = pd.DataFrame(rows).drop_duplicates(subset=["anchor"]).reset_index(drop=True)
    print(f"  Augmented: {len(aug_df)} total training pairs")
    return aug_df


# ─────────────────────────────────────────────────────────────
#  3. BUILD HF DATASETS
# ─────────────────────────────────────────────────────────────
def build_hf_datasets(aug_df: pd.DataFrame, eval_df: pd.DataFrame):
    train_hf = Dataset.from_dict({
        "anchor"  : aug_df["anchor"].tolist(),
        "positive": aug_df["positive"].tolist(),
    })
    eval_hf = Dataset.from_dict({
        "anchor"  : eval_df["roman_urdu_query"].tolist(),
        "positive": eval_df["english_translation"].tolist(),
    })
    return train_hf, eval_hf


# ─────────────────────────────────────────────────────────────
#  4. EVALUATOR
# ─────────────────────────────────────────────────────────────
def build_evaluator(eval_df: pd.DataFrame):
    return evaluation.EmbeddingSimilarityEvaluator(
        sentences1       = eval_df["roman_urdu_query"].tolist(),
        sentences2       = eval_df["english_translation"].tolist(),
        scores           = [1.0] * len(eval_df),
        name             = "savvymart_eval",
        show_progress_bar= False,
    )


# ─────────────────────────────────────────────────────────────
#  5. FINE-TUNE
# ─────────────────────────────────────────────────────────────
def fine_tune(df: pd.DataFrame):

    # ── Train / Eval split ───────────────────────────────────
    train_df, eval_df = train_test_split(
        df,
        test_size    = CONFIG["eval_split"],
        stratify     = df["category"],
        random_state = CONFIG["seed"],
    )
    print(f"\n  Train : {len(train_df)} rows")
    print(f"  Eval  : {len(eval_df)} rows")

    # ── Augment ──────────────────────────────────────────────
    print("\n  Augmenting...")
    aug_df = augment_dataframe(train_df)

    # ── HF Datasets ──────────────────────────────────────────
    train_hf, eval_hf = build_hf_datasets(aug_df, eval_df)

    # ── Load model ───────────────────────────────────────────
    print(f"\n  Loading : {CONFIG['base_model']}")
    model = SentenceTransformer(CONFIG["base_model"])
    model.max_seq_length = 64       # short queries — 64 tokens is plenty, faster training

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device  : {device.upper()}")
    if device == "cpu":
        print("  NOTE: Training on CPU will be slow (~30–60 min).")
        print("        Use Google Colab (free GPU) for faster training.")

    # ── Loss ─────────────────────────────────────────────────
    # MultipleNegativesRankingLoss — best for (query, product) pairs
    # Every other item in the batch acts as a hard negative
    train_loss = losses.MultipleNegativesRankingLoss(model)

    # ── Evaluator ────────────────────────────────────────────
    evaluator = build_evaluator(eval_df)

    # ── Compute steps ────────────────────────────────────────
    steps_per_epoch = math.ceil(len(aug_df) / CONFIG["batch_size"])
    total_steps     = steps_per_epoch * CONFIG["epochs"]
    warmup_steps    = math.ceil(total_steps * CONFIG["warmup_ratio"])

    print(f"\n  Training Plan")
    print(f"  {'─'*35}")
    print(f"  Epochs         : {CONFIG['epochs']}")
    print(f"  Batch size     : {CONFIG['batch_size']}")
    print(f"  Train pairs    : {len(aug_df)}")
    print(f"  Steps/epoch    : {steps_per_epoch}")
    print(f"  Total steps    : {total_steps}")
    print(f"  Warmup steps   : {warmup_steps}")
    print(f"  Learning rate  : {CONFIG['learning_rate']}")
    print(f"  {'─'*35}")

    # ── Training Arguments ───────────────────────────────────
    args = SentenceTransformerTrainingArguments(
        output_dir                  = CONFIG["output_dir"],
        num_train_epochs            = CONFIG["epochs"],
        per_device_train_batch_size = CONFIG["batch_size"],
        per_device_eval_batch_size  = CONFIG["batch_size"],
        learning_rate               = CONFIG["learning_rate"],
        warmup_steps                = warmup_steps,
        # NO_DUPLICATES: ensures anchor & positive never appear
        # in the same batch as each other's negatives
        batch_sampler               = BatchSamplers.NO_DUPLICATES,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_savvymart_eval_spearman_cosine",
        fp16                        = torch.cuda.is_available(),
        seed                        = CONFIG["seed"],
        logging_steps               = steps_per_epoch,
        report_to                   = "none",
    )

    # ── Trainer ──────────────────────────────────────────────
    trainer = SentenceTransformerTrainer(
        model         = model,
        args          = args,
        train_dataset = train_hf,
        eval_dataset  = eval_hf,
        loss          = train_loss,
        evaluator     = evaluator,
    )

    # ── Train ────────────────────────────────────────────────
    print(f"\n  Starting fine-tuning...\n  {'─'*35}")
    start   = datetime.now()
    trainer.train()
    elapsed = round((datetime.now() - start).total_seconds())

    # ── Save best model ──────────────────────────────────────
    save_path = CONFIG["output_dir"] + "/final"
    model.save_pretrained(save_path)

    print(f"\n  {'─'*35}")
    print(f"  Done in   : {elapsed}s ({elapsed//60}m {elapsed%60}s)")
    print(f"  Saved to  : ./{save_path}/")

    return model, eval_df


# ─────────────────────────────────────────────────────────────
#  6. EVALUATE — per-category cosine similarity
# ─────────────────────────────────────────────────────────────
def evaluate_model(model: SentenceTransformer, eval_df: pd.DataFrame) -> dict:
    print(f"\n  Evaluation Results")
    print(f"  {'─'*52}")
    print(f"  {'Category':<15} {'Avg Sim':>10} {'Min':>8} {'Max':>8}")
    print(f"  {'─'*52}")

    results = {}
    for cat, grp in eval_df.groupby("category"):
        ru   = model.encode(grp["roman_urdu_query"].tolist(),    show_progress_bar=False)
        en   = model.encode(grp["english_translation"].tolist(), show_progress_bar=False)
        sims = [cos_sim([ru[i]], [en[i]])[0][0] for i in range(len(ru))]
        avg, mn, mx = np.mean(sims), np.min(sims), np.max(sims)
        results[cat] = {"avg": round(float(avg),4), "min": round(float(mn),4), "max": round(float(mx),4)}
        bar = "█" * int(avg * 20)
        print(f"  {cat:<15} {avg:>10.4f} {mn:>8.4f} {mx:>8.4f}  {bar}")

    overall = np.mean([v["avg"] for v in results.values()])
    print(f"  {'─'*52}")
    print(f"  {'Overall':<15} {overall:>10.4f}")
    print(f"  {'─'*52}")
    return results


# ─────────────────────────────────────────────────────────────
#  7. DEMO SEARCH — short ecommerce queries like real users
# ─────────────────────────────────────────────────────────────
def demo_search(model: SentenceTransformer, df: pd.DataFrame):
    print(f"\n  Demo: SavvyMart Semantic Search")
    print(f"  {'='*60}")

    products     = df["english_translation"].drop_duplicates().tolist()
    product_embs = model.encode(products, show_progress_bar=False, convert_to_tensor=True)

    # Short, realistic ecommerce queries
    test_queries = [
        # Clothes
        "kali shirt",
        "safaid kurta",
        "nili jeans",
        "sasta shalwar kameez",
        "ladies kameez",
        # Bags
        "kala backpack",
        "safaid ladies bag",
        "sasta laptop bag",
        "travel trolley bag",
        # Electronics
        "sasta mobile",
        "wireless earphones",
        "gaming laptop",
        "fast charger",
        # Shoes
        "kali jooti",
        "safaid sneakers",
        "sasti chappal",
        # Hair Oil
        "balo ka tel",
        "dandruff oil",
        "sasta amla tel",
    ]

    for query in test_queries:
        q_emb = model.encode(query, convert_to_tensor=True)
        hits  = util.semantic_search(q_emb, product_embs, top_k=3)[0]
        print(f"\n  Query : \"{query}\"")
        for i, hit in enumerate(hits, 1):
            prod  = products[hit["corpus_id"]]
            score = hit["score"]
            flag  = " ✓" if score >= 0.75 else (" ~" if score >= 0.5 else " ✗")
            print(f"  {i}.{flag} {prod:<40}  score: {score:.4f}")


# ─────────────────────────────────────────────────────────────
#  8. SAVE LOG
# ─────────────────────────────────────────────────────────────
def save_log(eval_results: dict):
    log = {
        "timestamp"    : datetime.now().isoformat(),
        "model"        : CONFIG["base_model"],
        "dataset"      : CONFIG["dataset_path"],
        "config"       : CONFIG,
        "eval_results" : eval_results,
    }
    with open(CONFIG["log_path"], "w") as f:
        json.dump(log, f, indent=2)
    print(f"\n  Log saved : {CONFIG['log_path']}")


# ─────────────────────────────────────────────────────────────
#  9. PRODUCTION CLASS — plug into SavvyMart backend
# ─────────────────────────────────────────────────────────────
class SavvyMartSearch:
    """
    Semantic search engine for SavvyMart.
    Handles Roman Urdu, English, and mixed queries.

    Quick start:
        searcher = SavvyMartSearch("savvymart_model/final")
        searcher.index_products(product_name_list)
        results  = searcher.search("kali shirt", top_k=5)
    """

    def __init__(self, model_path: str = "savvymart_model/final"):
        print(f"  Loading model: {model_path}")
        self.model              = SentenceTransformer(model_path)
        self.product_embeddings = None
        self.products           = []
        print(f"  Model ready!")

    def index_products(self, products: list[dict], text_key: str = "name"):
        """
        Index your product catalog.

        Args:
            products : list of dicts from your DB
                       e.g. [{"id":1, "name":"Black Shirt", "price":500}, ...]
            text_key : which field to embed (default: "name")
        """
        self.products    = products
        product_texts    = [p[text_key] for p in products]
        print(f"  Indexing {len(products)} products...")
        self.product_embeddings = self.model.encode(
            product_texts,
            show_progress_bar=True,
            convert_to_tensor=True,
            batch_size=128,
        )
        print(f"  Indexed!")

    def search(self, query: str, top_k: int = 10, min_score: float = 0.3) -> list[dict]:
        """
        Search products.

        Args:
            query     : Roman Urdu or English query string
            top_k     : max results to return
            min_score : minimum similarity threshold (0–1)

        Returns:
            list of {rank, product, score}
        """
        if self.product_embeddings is None:
            raise RuntimeError("Call index_products() first.")

        q_emb = self.model.encode(query, convert_to_tensor=True)
        hits  = util.semantic_search(q_emb, self.product_embeddings, top_k=top_k)[0]

        return [
            {
                "rank"   : i + 1,
                "product": self.products[h["corpus_id"]],
                "score"  : round(h["score"], 4),
            }
            for i, h in enumerate(hits)
            if h["score"] >= min_score
        ]


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("  SavvyMart — Roman Urdu BERT Fine-Tuning  (v3)")
    print("=" * 55)

    # 1. Load
    df = load_dataset_from_excel(CONFIG["dataset_path"])

    # 2. Fine-tune
    model, eval_df = fine_tune(df)

    # 3. Evaluate per category
    eval_results = evaluate_model(model, eval_df)

    # 4. Demo with short real-world queries
    demo_search(model, df)

    # 5. Save training log
    save_log(eval_results)

    print("\n" + "=" * 55)
    print("  DONE! Use in your Flask/Django backend:")
    print("=" * 55)
    print("""
  from finetune_roman_urdu import SavvyMartSearch

  # Run ONCE at server startup
  searcher = SavvyMartSearch("savvymart_model/final")
  searcher.index_products(products_from_db, text_key="name")

  # On every search request
  results = searcher.search("kali shirt", top_k=10)
  for r in results:
      print(r["rank"], r["product"]["name"], r["score"])
    """)
